# Strategy Design Pattern 

explained using the classic Navigation Map example.

#### The Concept

The Strategy Pattern allows you to define a family of algorithms, put each of them into a separate class/function, and make their objects interchangeable. 

**Analogy**: Google Maps. When you want to go from Point A to Point B, you can choose a strategy:

- **Walking Strategy**: Avoids highways, prefers parks.
- **Driving Strategy**: Prefers highways, avoids traffic.
- **Cycling Strategy**: Prefers bike lanes.

The "Map" (Context) doesn't care how the route is calculated. It just asks the selected Strategy to "Build Route".

## The Classic OOP Way (Java-Style)

In the strict OOP approach, we define a common `RouteStrategy` interface. Every specific algorithm must implement this interface. The `Navigator` context holds a reference to the active strategy object.

#### THE STRATEGY INTERFACE

In [1]:
from abc import ABC, abstractmethod

class RouteStrategy(ABC):
    @abstractmethod
    def build_route(self, start: str, end: str):
        pass

#### CONCRETE STRATEGIES

In [2]:
class WalkingStrategy(RouteStrategy):
    def build_route(self, start: str, end: str):
        print(f"🚶 Calculating walking route from {start} to {end} via parks.")

class DrivingStrategy(RouteStrategy):
    def build_route(self, start: str, end: str):
        print(f"🚗 Calculating driving route from {start} to {end} via highway.")

class PublicTransportStrategy(RouteStrategy):
    def build_route(self, start: str, end: str):
        print(f"🚌 Calculating bus route from {start} to {end}.")

#### THE CONTEXT (Navigator)

In [3]:
class Navigator:
    def __init__(self, strategy: RouteStrategy):
        self._strategy = strategy

    def set_strategy(self, strategy: RouteStrategy):
        """Allows swapping strategies at runtime"""
        self._strategy = strategy

    def build_route(self, start: str, end: str):
        self._strategy.build_route(start, end)

#### CLIENT CODE

In [4]:
def main():
    # 1. Start with Walking
    nav = Navigator(WalkingStrategy())
    nav.build_route("Home", "Park")

    # 2. Switch to Driving (Runtime swap)
    print("--- User selected Car mode ---")
    nav.set_strategy(DrivingStrategy())
    nav.build_route("Home", "Work")

if __name__ == "__main__":
    main()

🚶 Calculating walking route from Home to Park via parks.
--- User selected Car mode ---
🚗 Calculating driving route from Home to Work via highway.


## The Pythonic Way (Functions as First-Class Citizens)

In Python, functions are objects. We don't need to write a whole class (`class WalkingStrategy`) just to hold one method (`build_route`). We can simply define functions and pass them directly to the Navigator.

#### THE STRATEGIES (Just Functions)

In [5]:
def strategy_walking(start: str, end: str):
    print(f"🚶 Walking route: {start} -> {end} (scenic path)")

def strategy_driving(start: str, end: str):
    print(f"🚗 Driving route: {start} -> {end} (fastest highway)")

def strategy_bus(start: str, end: str):
    print(f"🚌 Bus route: {start} -> {end} (Line 42)")

#### THE CONTEXT

In [6]:
from typing import Callable

class Navigator:
    def __init__(self, strategy_func: Callable[[str, str], None]):
        self.strategy = strategy_func

    def execute(self, start, end):
        # We call the function directly
        self.strategy(start, end)

#### CLIENT CODE

In [7]:
def main():
    # 1. Pass the function itself
    app = Navigator(strategy_walking)
    app.execute("Home", "Gym")

    # 2. Swap the function dynamically
    print("--- Changing to Bus ---")
    app.strategy = strategy_bus
    app.execute("Gym", "Airport")

    # 3. Even simpler: Using Lambda for one-off strategies
    print("--- Custom Strategy ---")
    app.strategy = lambda s, e: print(f"🚁 Flying from {s} to {e}!")
    app.execute("Airport", "Island")

if __name__ == "__main__":
    main()

🚶 Walking route: Home -> Gym (scenic path)
--- Changing to Bus ---
🚌 Bus route: Gym -> Airport (Line 42)
--- Custom Strategy ---
🚁 Flying from Airport to Island!


#### Key Differences

| Feature              | Classic OOP                                                     | Pythonic                                                     |
|----------------------|-----------------------------------------------------------------|---------------------------------------------------------------|
| **Strategy Definition** | Concrete class implementing an interface.                      | Simple function or lambda.                                   |
| **Boilerplate**        | High — requires interface, class, and method.                  | Low — just `def` or a lambda.                                |
| **State**              | Can hold internal state (e.g., a driving strategy storing data).| Can use closures or callable objects if state is needed.     |


#### When to use which?

- **Java Way**: If your strategy needs to maintain internal state (e.g., a caching mechanism inside the `DrivingStrategy` class) or has multiple methods (`calculate_cost`, `calculate_time`, `calculate_route`).
- **Python Way**: If your strategy is just a single algorithm or logic block. This is often used in sort keys (e.g., `list.sort(key=len)` is essentially using the Strategy pattern).

# Strategy Design Pattern 

explained using a complex, real-world scenario: E-Commerce Payment Processing.

#### The Scenario

In a large e-commerce system, you need to support multiple payment methods (Credit Card, PayPal, Crypto).
- **Credit Card**: Requires Card Number, CVV, Expiry. Needs strict validation (Luhn algorithm).
- **PayPal**: Requires Email and Password. Needs a simulated API login.
- **Bitcoin**: Requires a Wallet Address. Needs transaction hash verification.

Each method has completely different data requirements and validation logic, but they all share the same goal: **Process a Payment**.

## The Classic OOP Way (Java-Style)

We define a `PaymentStrategy` interface. Each specific payment method is a Class that implements this interface. Crucially, the class holds the specific State (like credit card numbers) required for that strategy to work.

#### THE STRATEGY INTERFACE

In [8]:
from abc import ABC, abstractmethod

class PaymentStrategy(ABC):
    @abstractmethod
    def pay(self, amount: float) -> bool:
        pass

#### CONCRETE STRATEGIES (Stateful Classes)

In [10]:
class CreditCardStrategy(PaymentStrategy):
    def __init__(self, name: str, card_num: str, cvv: str, expiry: str):
        # Specific state for Credit Cards
        self.name = name
        self.card_num = card_num
        self.cvv = cvv
        self.expiry = expiry

    def pay(self, amount: float) -> bool:
        # Complex validation logic specific to cards
        if len(self.cvv) != 3:
            print("❌ Transaction Failed: Invalid CVV.")
            return False
        print(f"💳 Paid ${amount} using Credit Card ({self.card_num[-4:]})")
        return True

class PayPalStrategy(PaymentStrategy):
    def __init__(self, email: str, password: str):
        # Specific state for PayPal
        self.email = email
        self.password = password

    def pay(self, amount: float) -> bool:
        # Complex logic specific to PayPal (simulated login)
        if "@" not in self.email:
            print("❌ Transaction Failed: Invalid Email.")
            return False
        print(f"🅿️ Paid ${amount} using PayPal Account ({self.email})")
        return True

#### THE CONTEXT (Shopping Cart)

In [11]:
class ShoppingCart:
    def __init__(self):
        self.total_amount = 0.0

    def add_item(self, price: float):
        self.total_amount += price

    def checkout(self, payment_method: PaymentStrategy):
        # The Context delegates the work to the Strategy object
        # It doesn't know if it's using Card or PayPal.
        if payment_method.pay(self.total_amount):
            print("✅ Checkout Successful!\n")
            self.total_amount = 0 # Reset cart
        else:
            print("❌ Checkout Failed.\n")

#### CLIENT CODE

In [12]:
def main():
    cart = ShoppingCart()
    cart.add_item(100.0)
    cart.add_item(50.0)

    # User chooses Credit Card
    # We must instantiate the specific class with specific data
    cc_strategy = CreditCardStrategy("John Doe", "1234-5678-9012-3456", "123", "12/25")
    cart.checkout(cc_strategy)

    # User chooses PayPal
    cart.add_item(25.0)
    paypal_strategy = PayPalStrategy("john@example.com", "secret123")
    cart.checkout(paypal_strategy)

if __name__ == "__main__":
    main()

💳 Paid $150.0 using Credit Card (3456)
✅ Checkout Successful!

🅿️ Paid $25.0 using PayPal Account (john@example.com)
✅ Checkout Successful!



## The Pythonic Way (First-Class Functions)

In Python, we don't always need a class to hold state. We can use Functions for the logic. But wait—what about the state (Card number vs Email)?
- **Closures**: We can capture state inside a function.
- **Partial Functions**: We can pre-fill arguments.

Here, I will show a very powerful Pythonic pattern: **The Strategy Registry**. Instead of if/else checks or instantiating classes manually, we map keys (strings) to functions.

#### THE STRATEGIES (Pure Functions)

In [13]:
def pay_via_cc(amount: float, card_num: str, cvv: str) -> bool:
    if len(cvv) != 3:
        print("❌ Invalid CVV")
        return False
    print(f"💳 Paid ${amount} via Credit Card {card_num[-4:]}")
    return True

def pay_via_paypal(amount: float, email: str) -> bool:
    if "@" not in email:
        print("❌ Invalid Email")
        return False
    print(f"🅿️ Paid ${amount} via PayPal ({email})")
    return True

def pay_via_bitcoin(amount: float, wallet: str) -> bool:
    print(f"🪙 Paid ${amount} via Bitcoin Wallet {wallet[:6]}...")
    return True

#### THE CONTEXT (Functional Dispatch)

In [14]:
from dataclasses import dataclass
from typing import Callable

@dataclass
class Order:
    amount: float
    
    # The 'strategy' is just a Callable function
    def process_payment(self, payment_func: Callable[..., bool], **kwargs):
        """
        Takes a function and whatever arguments that function needs (kwargs).
        """
        try:
            success = payment_func(self.amount, **kwargs)
            if success:
                print("✅ Payment Complete\n")
            else:
                print("❌ Payment Failed\n")
        except TypeError as e:
            # Handles case where wrong arguments are passed to the strategy
            print(f"❌ System Error: Missing payment details for this method. {e}\n")

#### CLIENT CODE

In [15]:
def main():
    order = Order(amount=150.00)

    print("--- 1. Using Credit Card Function ---")
    # We pass the function itself, and the arguments it needs
    order.process_payment(
        pay_via_cc, 
        card_num="4444-5555-6666-7777", 
        cvv="999"
    )

    print("--- 2. Using PayPal Function ---")
    order.process_payment(
        pay_via_paypal, 
        email="jane@example.com"
    )

    # --- ADVANCED: DYNAMIC DISPATCH ---
    # Imagine this comes from a dropdown menu in a UI
    selected_method = "bitcoin" 
    
    # Map strings to functions (The Registry Pattern)
    strategies = {
        "cc": pay_via_cc,
        "paypal": pay_via_paypal,
        "bitcoin": pay_via_bitcoin
    }
    
    user_data = {"wallet": "1A1zP1eP5QGefi2DMPTfTL5SLmv7DivfNa"}
    
    print(f"--- 3. Dynamic Selection ({selected_method}) ---")
    if selected_method in strategies:
        func = strategies[selected_method]
        order.process_payment(func, **user_data)

if __name__ == "__main__":
    main()

--- 1. Using Credit Card Function ---
💳 Paid $150.0 via Credit Card 7777
✅ Payment Complete

--- 2. Using PayPal Function ---
🅿️ Paid $150.0 via PayPal (jane@example.com)
✅ Payment Complete

--- 3. Dynamic Selection (bitcoin) ---
🪙 Paid $150.0 via Bitcoin Wallet 1A1zP1...
✅ Payment Complete



#### Key Differences & Why It Matters

- **State Management**:
  - **OOP**: The `CreditCardStrategy` object holds the data (`self.cvv`). It is a "stateful container".
  - **Pythonic**: The function `pay_via_cc` is stateless. The data (`cvv`) is passed in at the moment of execution via `**kwargs` or closures.

- **Extensibility**:
  - **OOP**: To add Bitcoin, you must create a `class BitcoinStrategy(PaymentStrategy)`.
  - **Pythonic**: You just write a function `def pay_via_bitcoin(...)`. You don't need to inherit from anything.

- **Flexibility**:
  - The Pythonic version using `**kwargs` allows different strategies to accept completely different parameters (one needs `cvv`, one needs `wallet`) without changing the `Order` class. In strict Java OOP, handling varying parameters in an interface usually requires a complex "PaymentDetails" DTO (Data Transfer Object) or casting.
 

# Strategy Design Pattern 

applied to a complex Bank Loan System.

#### The Scenario: Personalized Loan Offers

A bank needs to calculate the best loan offer (Amount, Interest, Tenure) for a customer. The calculation logic changes completely based on the customer's relationship with the bank:
- **Bank Only**: Logic depends on their **Average Balance**.
- **Card Only**: Logic depends on their **Credit Score** and **Card Limit**.
- **Bank + Card (Bundle)**: Logic mixes both and gives a **Loyalty Bonus** (Lower Interest).

## The Classic OOP Way (Java-Style)

We define a `LoanStrategy` interface. We create three separate classes for the calculation logic. The `Customer` object is passed to these strategies.

#### THE DATA (Customer)

In [16]:
from dataclasses import dataclass

@dataclass
class Customer:
    name: str
    cust_type: str  # "BANK", "CARD", "BOTH"
    balance: float = 0.0
    credit_score: int = 0
    card_limit: float = 0.0
    is_priority: bool = False

@dataclass
class LoanOffer:
    amount: float
    interest_rate: float
    tenure_months: int
    emi: float

    def __str__(self):
        return (f"Offer for {self.amount} @ {self.interest_rate}% "
                f"for {self.tenure_months} months (EMI: {self.emi})")

#### THE STRATEGY INTERFACE

In [19]:
from abc import ABC, abstractmethod

class LoanStrategy(ABC):
    @abstractmethod
    def calculate_offer(self, customer: Customer) -> LoanOffer:
        pass

#### CONCRETE STRATEGIES (The Algorithms)

In [21]:
class BankOnlyStrategy(LoanStrategy):
    """
    Logic: Loan is 5x the bank balance. 
    Interest is standard 10%.
    """
    def calculate_offer(self, customer: Customer) -> LoanOffer:
        amount = customer.balance * 5
        rate = 10.0
        
        # Priority customers get slightly better rate
        if customer.is_priority:
            rate -= 1.0
            
        months = 24
        emi = (amount * (1 + rate/100)) / months
        return LoanOffer(amount, rate, months, round(emi, 2))

class CardOnlyStrategy(LoanStrategy):
    """
    Logic: Loan is 2x the card limit. 
    Interest is high (14%) because we don't hold their money.
    Logic depends heavily on Credit Score.
    """
    def calculate_offer(self, customer: Customer) -> LoanOffer:
        if customer.credit_score < 700:
            return LoanOffer(0, 0, 0, 0) # No loan

        amount = customer.card_limit * 2.0
        rate = 14.0
        months = 12
        emi = (amount * (1 + rate/100)) / months
        return LoanOffer(amount, rate, months, round(emi, 2))

class BundleStrategy(LoanStrategy):
    """
    Logic: Best of both worlds.
    Loan = (Balance * 5) + (Limit * 2).
    Interest is lowest (8%) as Loyalty Bonus.
    """
    def calculate_offer(self, customer: Customer) -> LoanOffer:
        amount = (customer.balance * 5) + (customer.card_limit * 2)
        rate = 8.0 # Loyalty Benefit
        months = 36 # Longer tenure allowed
        emi = (amount * (1 + rate/100)) / months
        return LoanOffer(amount, rate, months, round(emi, 2))

#### THE CONTEXT (Loan Calculator)

In [22]:
class LoanService:
    def __init__(self, strategy: LoanStrategy):
        self.strategy = strategy

    def get_best_offer(self, customer: Customer):
        print(f"--- Calculating for {customer.name} ({customer.cust_type}) ---")
        offer = self.strategy.calculate_offer(customer)
        print(offer)
        print()

#### CLIENT CODE

In [23]:
def main():
    # 1. Create Customers
    alice = Customer("Alice", "BANK", balance=10000, is_priority=True)
    bob = Customer("Bob", "CARD", credit_score=750, card_limit=50000)
    charlie = Customer("Charlie", "BOTH", balance=20000, credit_score=800, card_limit=50000)

    # 2. Select Strategy and Execute
    
    # Alice (Bank Only)
    service = LoanService(BankOnlyStrategy())
    service.get_best_offer(alice)

    # Bob (Card Only)
    service = LoanService(CardOnlyStrategy())
    service.get_best_offer(bob)

    # Charlie (Bundle)
    service = LoanService(BundleStrategy())
    service.get_best_offer(charlie)

if __name__ == "__main__":
    main()

--- Calculating for Alice (BANK) ---
Offer for 50000 @ 9.0% for 24 months (EMI: 2270.83)

--- Calculating for Bob (CARD) ---
Offer for 100000.0 @ 14.0% for 12 months (EMI: 9500.0)

--- Calculating for Charlie (BOTH) ---
Offer for 200000 @ 8.0% for 36 months (EMI: 6000.0)



## The Pythonic Way (Strategy Registry)

In Python, we can remove the boilerplate classes. We define the algorithms as Functions. We also use a **Dictionary (Registry)** to automatically map the `customer_type` string (e.g., "BANK") to the correct function. This removes the need for `if/else` logic in the client code.

#### THE DATA

In [24]:
from dataclasses import dataclass

@dataclass
class Customer:
    name: str
    cust_type: str # Key for strategy selection
    balance: float = 0
    credit_score: int = 0
    limit: float = 0
    priority: bool = False

#### THE STRATEGIES (Pure Functions)

In [25]:
def strategy_bank(c: Customer) -> dict:
    """Uses Balance Logic"""
    amount = c.balance * 5
    rate = 9.0 if c.priority else 10.0
    return {"amount": amount, "rate": rate, "tenure": 24}

def strategy_card(c: Customer) -> dict:
    """Uses Credit Score Logic"""
    if c.credit_score < 700:
        return {"error": "Score too low"}
    
    amount = c.limit * 3
    rate = 14.0 # High risk
    return {"amount": amount, "rate": rate, "tenure": 12}

def strategy_bundle(c: Customer) -> dict:
    """Uses Hybrid Logic + Loyalty Bonus"""
    amount = (c.balance * 5) + (c.limit * 3)
    rate = 7.5 # Best rate
    return {"amount": amount, "rate": rate, "tenure": 48}

#### THE CONTEXT (Auto-Dispatcher)

In [26]:
from typing import Dict, Callable

class LoanEngine:
    """
    The engine doesn't need to be told which strategy to use manually.
    It looks it up in a registry based on the customer data itself.
    """
    # Mapping String Keys -> Functions
    _strategies: Dict[str, Callable] = {
        "BANK_ONLY": strategy_bank,
        "CARD_ONLY": strategy_card,
        "PREMIUM_BUNDLE": strategy_bundle
    }

    @classmethod
    def calculate(cls, customer: Customer):
        # 1. Lookup Strategy
        logic_func = cls._strategies.get(customer.cust_type)
        
        if not logic_func:
            print(f"❌ No loan policy found for type: {customer.cust_type}")
            return

        # 2. Execute
        result = logic_func(customer)
        
        # 3. Print Result
        print(f"Loan Offer for {customer.name} ({customer.cust_type}):")
        if "error" in result:
            print(f"   -> Declined: {result['error']}")
        else:
            print(f"   -> Amount: ${result['amount']}")
            print(f"   -> Rate:   {result['rate']}%")
            print(f"   -> Tenure: {result['tenure']} months")
        print("-" * 30)

#### CLIENT CODE

In [27]:
def main():
    # Notice we use the keys "BANK_ONLY", "CARD_ONLY" directly in data
    c1 = Customer("John", "BANK_ONLY", balance=5000, priority=True)
    c2 = Customer("Jane", "CARD_ONLY", credit_score=650, limit=10000) # Low score
    c3 = Customer("Max",  "PREMIUM_BUNDLE", balance=50000, limit=200000)
    c4 = Customer("Unknown", "CRYPTO_USER") # Edge case

    # The Engine handles the dispatch automatically
    LoanEngine.calculate(c1)
    LoanEngine.calculate(c2)
    LoanEngine.calculate(c3)
    LoanEngine.calculate(c4)

if __name__ == "__main__":
    main()

Loan Offer for John (BANK_ONLY):
   -> Amount: $25000
   -> Rate:   9.0%
   -> Tenure: 24 months
------------------------------
Loan Offer for Jane (CARD_ONLY):
   -> Declined: Score too low
------------------------------
Loan Offer for Max (PREMIUM_BUNDLE):
   -> Amount: $850000
   -> Rate:   7.5%
   -> Tenure: 48 months
------------------------------
❌ No loan policy found for type: CRYPTO_USER


#### Why the Pythonic version fits this example perfectly

- **Registry Pattern**: In the OOP version, the `main()` function had to manually decide: "If customer is bank, use `BankStrategy`". In the Pythonic version, the `LoanEngine` uses a Dictionary `_strategies`. This means you can add new loan types (e.g., "`MORTGAGE"`) just by adding one line to the dictionary, without changing the engine code.
- **Cleaner Data Flow**: The strategies return simple dictionaries. This is often easier to pass to a Frontend API (JSON) than Java-style objects.